# Notebook 6: Running Your Own Language Model (Arabic and English)

For Digital Humanities students. No programming experience needed.

**You will learn**
1. What an open-source model is
2. How to ask it questions in English and Arabic
3. How to control it (system prompt, temperature, top-p, top-k)
4. How to teach it by example and get tidy tables back

**Open-source model:** a model you download and run yourself, instead of sending your text to a company (as with ChatGPT). Here it runs on a free GPU from Google Colab. It is free and your text stays in your session, but answers are weaker than the big services.

**How to use this notebook:** click a grey code cell and press Shift+Enter. Run cells from top to bottom. Lines starting with `#` are comments for humans; Python ignores them.

## Setup

**1. Turn on the free GPU.** Menu: Runtime > Change runtime type > T4 GPU > Save.

**2. Hugging Face.** Hugging Face is a free website where researchers share AI models, like an app store for AI. We download our model from there.

We use **Qwen**, which is good at Arabic and **needs no account and no token**. Just run the cells below.

*Optional:* to use Meta's Llama instead, you need a free Hugging Face account, you must accept Meta's licence on the model page, and you must create a "Read" token at huggingface.co/settings/tokens. Then save it in Colab (key icon in the left sidebar) with the name `HF_TOKEN`. Llama is weaker in Arabic, so we recommend skipping this.

In [ ]:
# Check that we have a GPU, and install the tools we need.
import torch
print("GPU found:" if torch.cuda.is_available() else "No GPU. Turn it on in Runtime > Change runtime type.",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# "!" runs a system command. This installs the libraries (takes about a minute).
!pip install -q -U transformers accelerate arabic-reshaper python-bidi

In [ ]:
import json
import numpy as np                      # numbers
import pandas as pd                     # tables
import matplotlib.pyplot as plt         # charts
from transformers import AutoTokenizer, AutoModelForCausalLM

# The model's name on Hugging Face. Qwen needs no token.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # Llama: needs the token (optional)

# The tokenizer cuts text into small pieces ("tokens") the model can read.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The model itself. The first run downloads about 6 GB (a few minutes).
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
print("Model loaded:", MODEL_ID)

### The helper function `ask()`

To keep things short, one function does all the technical work. You give it a question, it gives back an answer. Optional settings: `system`, `temperature`, `top_p`, `top_k` (explained below).

In [ ]:
def ask(prompt, system=None, temperature=None, top_p=None, top_k=None, max_tokens=300):
    """Send a question to the model and return its answer as text."""
    # Build the conversation, like a chat app.
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    # Turn the conversation into tokens and move them to the GPU.
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

    # Randomness settings. If none are given, the model always picks its most likely word.
    settings = dict(max_new_tokens=max_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature != 0 and (temperature or top_p or top_k):
        settings.update(do_sample=True, temperature=temperature or 1.0,
                        top_p=top_p or 1.0, top_k=top_k or 0)
    else:
        settings.update(do_sample=False, temperature=None, top_p=None, top_k=None)

    with torch.no_grad():
        output = model.generate(**inputs, **settings)

    # Keep only the new words (not our question) and turn them back into text.
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask("Reply with one word: ready", max_tokens=10))

## 1. Your first prompts

A **prompt** is the text you send to the model. Same question, in English and in Arabic.

In [ ]:
print("English:", ask("In two sentences, explain what the Digital Humanities are."))
print()
print("Arabic: ", ask("في جملتين، اشرح ما هي العلوم الإنسانية الرقمية."))

Notice: the Arabic answer may be shorter or weaker. Small models have seen less Arabic than English. Always read the output critically.

## 2. System prompt: give the model a role

A **system prompt** is a standing instruction: who the model is, who it talks to, how long to answer.

In [ ]:
question = "What is OCR, and why does it make mistakes on old printed texts?"

print("--- No system prompt ---")
print(ask(question))

print("\n--- Tutor, 3 short bullets ---")
print(ask(question, system="You are a friendly tutor for first-year Digital Humanities students. "
                           "Answer in exactly 3 short bullet points. Avoid jargon."))

# The same idea in Arabic
print("\n--- Arabic tutor ---")
print(ask("ما هو التعرف الضوئي على الحروف (OCR)، ولماذا يخطئ في النصوص المطبوعة القديمة؟",
          system="أنت مدرّس لطيف لطلاب العلوم الإنسانية الرقمية. أجب بالعربية في ثلاث نقاط قصيرة."))

**Lesson:** say who the reader is, what format you want, and how long. Short, clear instructions work best.

## 3. Temperature: how creative is the model?

The model writes one small piece of text (a **token**, about a word) at a time. For each step it gives every possible next word a probability, then picks one.

- **Low temperature (0.2):** picks the likely words. Consistent. Good for extracting data.
- **High temperature (1.2):** gives unlikely words a chance. Varied. Good for brainstorming.

In [ ]:
prompts = {
    "English": "Suggest a title for an article about digitising Arabic manuscripts. Give only the title.",
    "Arabic":  "اقترح عنوانًا لمقالة عن رقمنة المخطوطات العربية. اذكر العنوان فقط.",
}

for lang, prompt in prompts.items():
    for temp in [0.2, 1.2]:
        print(f"{lang}, temperature {temp}")
        for _ in range(3):                      # ask 3 times to see the variety
            print("  -", ask(prompt, temperature=temp, max_tokens=40))

### Seeing it: the model's word probabilities

Because we run the model ourselves, we can look inside. The charts show the 10 words the model finds most likely after a sentence start, at three temperatures. Run the cell, then compare English and Arabic.

In [ ]:
# Chart helpers. Just run this cell; you do not need to read it.
import re
import arabic_reshaper
from bidi.algorithm import get_display

def fix_ar(text):
    """Make Arabic readable in charts (join the letters and write right-to-left)."""
    if re.search(r"[\u0600-\u06FF]", text):
        return get_display(arabic_reshaper.reshape(text))
    return text

def get_scores(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        return model(**inputs).logits[0, -1].float().cpu()    # scores for the next word

def draw(ax, scores, temperature=1.0, top_k=None, top_p=None, title=""):
    probs = torch.softmax(scores / temperature, dim=-1)       # temperature reshapes the probabilities
    best = torch.topk(probs, 10)
    words = [fix_ar(tokenizer.decode(int(i)).strip() or "(space)") for i in best.indices]
    values = best.values.numpy()
    keep = np.ones(10, dtype=bool)
    if top_k: keep[top_k:] = False
    if top_p: keep[int(np.searchsorted(np.cumsum(values), top_p)) + 1:] = False
    ax.bar(range(10), values * 100, color=["#2a6f97" if k else "#c9c9c9" for k in keep])
    ax.set_xticks(range(10)); ax.set_xticklabels(words, rotation=60, ha="right")
    ax.set_ylabel("probability (%)"); ax.set_title(title)

starts = {"English": "The manuscript was written in", "Arabic": "كُتبت المخطوطة باللغة"}
all_scores = {lang: get_scores(text) for lang, text in starts.items()}

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for row, lang in enumerate(starts):
    for col, temp in enumerate([0.5, 1.0, 2.0]):
        draw(axes[row][col], all_scores[lang], temperature=temp, title=f"{lang}, temperature {temp}")
plt.tight_layout(); plt.show()

**What to look for:** at 0.5 the top bars dominate. At 2.0 the bars are flatter, so surprising words get a chance. In Arabic the bars are often more spread out: the model is less sure. Note that the model works with pieces of words, so some Arabic labels are short fragments (for example the prefix "ال", meaning "the"), not whole words.

## 4. Top-k and top-p: shorten the list of candidates

- **Top-k:** keep only the k most likely words. `top_k=3` keeps exactly three.
- **Top-p:** keep the smallest group of words whose probabilities add up to p. `top_p=0.6` keeps as many words as needed to reach 60%.

Use `ask(prompt, top_k=3)` or `ask(prompt, top_p=0.6)`. Change only one setting at a time. For reproducible research, use `temperature=0` and write it in your methods.

Blue bars below are allowed, grey bars are removed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
draw(axes[0], all_scores["English"], title="No limit")
draw(axes[1], all_scores["English"], top_k=3, title="top_k = 3")
draw(axes[2], all_scores["English"], top_p=0.6, title="top_p = 0.6")
plt.tight_layout(); plt.show()

## 5. Few-shot prompting: teach by example

Instead of only describing the task, show a few solved examples. Small models follow patterns well.

Task: label the document type of archive snippets (a cataloguer's job).

In [ ]:
few_shot = """Classify each snippet as one of: LETTER, NEWSPAPER, POEM, LEGAL_DEED.
Answer with the label only.

Text: "Dear Mother, I hope this letter finds you well. The voyage was long."
Label: LETTER

Text: "The city council announced on Tuesday that the old market will be restored."
Label: NEWSPAPER

Text: "I, the undersigned, hereby sell to Ahmad ibn Ali the house for two hundred dinars."
Label: LEGAL_DEED

Text: "The moon walks softly over the sleeping sea."
Label:"""

print("English snippet:", ask(few_shot, temperature=0, max_tokens=8))

# Now the same task on Arabic snippets. Instructions stay in English.
arabic = {
    "عزيزتي أمي، أرجو أن يصلك هذا الخطاب وأنتِ بخير.":          "LETTER",
    "أعلن مجلس المدينة عن ترميم السوق القديم.":                  "NEWSPAPER",
    "اشترى أحمد بن علي الدار في الحي القديم بمائتي دينار.":      "LEGAL_DEED",
    "يمشي القمرُ رويدًا فوقَ البحرِ النائم.":                     "POEM",
}

rows = []
for text, expected in arabic.items():
    prompt = few_shot.rsplit("Text:", 1)[0] + f'Text: "{text}"\nLabel:'   # swap in the Arabic text
    answer = ask(prompt, temperature=0, max_tokens=8).split()[0]
    rows.append({"Arabic text": text, "Expected": expected, "Model said": answer,
                 "Correct": "yes" if answer == expected else "no"})
pd.DataFrame(rows)

Check every answer yourself. Small models are less reliable in Arabic, and your methods section should say so.

## 6. Structured output: from text to a table

Researchers need tables, not paragraphs. We ask for **JSON** (a simple, standard data format) and Python turns it into a table. Showing one example of the format makes small models much more reliable.

In [ ]:
descriptions = [
    "Letter from Mary Hall to her brother Thomas, written in Alexandria on 14 March 1887. "
    "Four pages, in English, held in a private family collection.",
    "رسالة من ليلى حسن إلى أخيها خالد، كُتبت في القدس بتاريخ 12 أيار 1952. "
    "ثلاث صفحات باللغة العربية، محفوظة في أرشيف العائلة.",
]

def build_prompt(description):
    return ('Extract catalogue metadata. Return ONLY valid JSON with the keys "author", "recipient", '
            '"place", "date", "pages", "language". Use null if a value is not stated.\n\n'
            'Example description: Poem by Layla Hassan, copied in Cairo in 1950, two pages, in Arabic.\n'
            'Example answer: {"author": "Layla Hassan", "recipient": null, "place": "Cairo", '
            '"date": "1950", "pages": 2, "language": "Arabic"}\n\n'
            f'Description: {description}\nAnswer:')

rows = []
for d in descriptions:
    raw = ask(build_prompt(d), temperature=0, max_tokens=200)
    try:
        rows.append(json.loads(raw[raw.find("{"): raw.rfind("}") + 1]))   # keep only the {...} part
    except ValueError:
        rows.append({"author": "(model wrote invalid JSON, run again)"})

pd.DataFrame(rows)

You now have a table you can sort, count, or save with `df.to_csv("metadata.csv")`. If a row shows an error, run the cell again. This is normal for small models, and the reason to always check results.

## Good habits

- **Verify.** Models can state false things confidently.
- **Record** the prompt, `MODEL_ID`, settings, and date for your methods section.
- **Test on your own language and material** before trusting results.
- **Confidential data:** Colab runs on Google's computers. Use your own laptop or university server for sensitive texts.

## Exercises

1. In section 2, change the role to "a museum guide for children". Try English and Arabic.
2. In section 3, change the sentence starts (for example "The scribe copied the text in") and rerun the chart.
3. In section 5, add a fifth label such as `MAP_DESCRIPTION` with one example.
4. In section 6, add a description from your own research material.